<a href="https://colab.research.google.com/github/ju4n1t0x/subtes/blob/main/Prediccion_fallas_subte_G11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Predicción de fallas en las formaciones de la red de subterráneos de Buenos Aires

A partir de su desgaste acumulado, como insumo para la planificación del mantenimiento preventivo.

---

## 📂 Repositorio del proyecto
Todo el desarrollo, los datasets crudos y el dataset limpio están en el repositorio de GitHub:
 **[prediccion_fallas_subte](https://github.com/ju4n1t0x/subtes)**


---

## 📊  Fuentes de datos
Se utilizan conjuntos de datos públicos relacionados con la operación de la red de subterráneos de Buenos Aires. Los archivos descargados y preparados para este trabajo se encuentran reunidos en la carpeta Datasets del repositorio.

Las bases incluyen registros de formaciones despachadas, estado de la flota, cantidad de pasajeros, estaciones, accesibilidad y trazado de las líneas. En la sección 2 se describe el contenido y el rol de cada conjunto de datos.



# 1. 📌 Introducción
**Descripción del tema**

El servicio de subterráneos de Buenos Aires depende de una flota acotada de formaciones que circulan de manera intensiva. El desgaste acumulado provoca fallas que retiran unidades de circulación, reducen la capacidad de servicio y amplían los intervalos entre trenes. Actualmente, estas fallas se atienden cuando ya se produjeron, ya que el operador no dispone de una herramienta que estime qué formaciones se encuentran próximas a fallar.

La información pública disponible permite reconstruir la historia de uso de cada formación a partir de los registros de despacho, kilómetros recorridos, fallas, composición de la flota, frecuencia y volumen de pasajeros. Utilizar estos datos de manera conjunta permitiría anticipar el deterioro y programar las intervenciones en ventanas de menor impacto sobre el servicio, en lugar de asumir la falla como una contingencia.

**Objetivo General**

Desarrollar un modelo predictivo de aprendizaje automático que estime el riesgo de falla de las formaciones de la red de subterráneos de Buenos Aires en función de su desgaste acumulado y de sus condiciones de operación, a fin de anticipar las unidades que requieren intervención y aportar criterios para la programación del mantenimiento preventivo.


**Hipótesis**

Los antecedentes en mantenimiento predictivo ferroviario muestran que las fallas del material rodante pueden anticiparse con niveles de precisión elevados. Aunque en la red de subterráneos de Buenos Aires no se dispone públicamente de datos de sensores, sí existe un historial operacional de cada formación.

Se plantea que la ocurrencia de fallas está asociada al desgaste acumulado, al modelo de flota y a la intensidad de servicio. **Se espera que los registros operacionales permitan estimar el riesgo de falla de cada formación con anticipación útil para la planificación del mantenimiento, incluso sin disponer de datos de sensores.**


# 2. 📊 Fuentes y conjuntos de datos

Los conjuntos de datos utilizados en el proyecto están almacenados en la carpeta `Datasets/` del repositorio de GitHub.

Se seleccionaron archivos con distintos niveles de detalle y períodos temporales. Las bases incluyen registros de formaciones despachadas, estado de la flota, cantidad de pasajeros, estaciones, accesibilidad y trazado de las líneas de la red de subterráneos de Buenos Aires.

Repositorio: [subtes – GitHub](https://github.com/ju4n1t0x/subtes)

## 2.1. Descripción de las bases utilizadas

| Base | Qué aporta | Filas × columnas | Rol en el trabajo |
|---|---|---:|---|
| `formaciones-despachadas-total.csv` | Registros históricos de despacho: fecha, línea, tipo de día, registro, orden, tren, datos de coches, kilómetros, tipo de viaje y horarios de salida. | 751.351 × 18 | Información operacional histórica de las formaciones (2015–2021). |
| `formaciones-despachadas-2025.csv` | Registros de despacho con formación, modelo, causa, cantidad de coches, kilómetros, tipo de viaje y hora de salida para los lados A y D. | 471.023 filas no vacías × 20 | Información operacional de 2025. |
| `formaciones-despachadas-2026.csv` | Estructura similar a la base de 2025, con datos de formación, modelo, causa, coches, kilómetros, tipo de viaje y horario. | 349.166 × 20 | Información operacional de 2026 (hasta agosto en el archivo disponible). |
| `estado-de-flota.csv` | Estado mensual de la flota por línea y modelo: unidades en servicio, reserva, RG, mantenimiento/estacionamiento y alistamiento. | 330 × 9 | Contexto sobre disponibilidad y composición de la flota (2024–2026). |
| `BaseUnificadaEstaciones.csv` | Cantidad de pasajeros por fecha, estación, línea y horario. | 822.363 × 6 | Contexto de demanda y utilización de estaciones (2020). |
| `total_viajes_por_linea.csv` | Pasajeros pagos, pases pagos, franquicias y total, agregados por período mensual y línea. | 576 × 7 | Contexto de demanda mensual por línea (2014–2021). |
| `viajes_anual.csv` | Total anual de viajes por línea. | 48 × 3 | Contexto de demanda anual por línea (2013–2020). |
| `estaciones-accesibles.csv` | Coordenadas, línea y estación; cantidad de escaleras mecánicas y ascensores. | 93 × 6 | Información complementaria de infraestructura y accesibilidad. |
| `lineas-de-subte.csv` | Geometría del trazado de las líneas, identificador y nombre de línea. | 82 × 3 | Información geográfica complementaria de la red. |

**Aclaración sobre las dimensiones:** en `formaciones-despachadas-2025.csv` se contabilizan 471.023 filas con algún dato; el archivo contiene además filas vacías. Las dimensiones corresponden a los archivos incluidos en el repositorio, no a la totalidad de los registros originales publicados por los organismos de origen.

**Períodos disponibles:** las bases no cubren todas el mismo período. En particular, los registros históricos de formaciones llegan hasta octubre de 2021; hay archivos separados para 2025 y 2026. Por eso, antes de integrar las bases, será necesario revisar la compatibilidad de sus columnas, identificadores y períodos.

## 2.2. Preparación y agregación de los datos

*El repositorio incluye el script `main.py`, utilizado para generar `total_viajes_por_linea.csv`. El script toma los registros históricos de pasajeros de 2014 a 2021, unifica los nombres de las columnas y normaliza los nombres de las líneas. Luego agrupa los registros por año, mes y línea, sumando las categorías de pasajeros. El resultado contiene 576 registros y 7 columnas*.

*Los archivos históricos individuales de pasajeros utilizados por el script no están incluidos en la carpeta actual del repositorio; sí está incluido el archivo agregado resultante (`total_viajes_por_linea.csv`). Por lo tanto, para este trabajo se documenta y utiliza el archivo agregado disponible.*

Las bases de formaciones de 2015–2021, 2025 y 2026 se conservan como archivos separados en el repositorio. No se asume que sean directamente concatenables: primero se deben comparar sus campos y criterios de registro para definir una estructura común, si corresponde.

# 3. Carga de datos

In [5]:
# Importación de librerías necesarias para la manipulación, el análisis y la visualización de los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 3.2. Carga de los conjuntos de datos

Se importan los archivos seleccionados desde el repositorio de GitHub mediante la biblioteca Pandas. Cada conjunto de datos se almacena en un DataFrame para facilitar su exploración y posterior análisis.

Los conjuntos de datos se organizan en dos grupos según el tipo de información que contienen. Por un lado, se cargan los registros operativos de las formaciones correspondientes a distintos períodos. Estos archivos se mantienen separados inicialmente para verificar sus estructuras y evaluar si es posible integrarlos. Por otro lado, se cargan los conjuntos de datos complementarios, que aportan información sobre el estado de la flota, la cantidad de pasajeros, las líneas y las estaciones.

In [6]:
# Carga de los registros operativos de las formaciones correspondientes a 2025.

url_2025 = "https://raw.githubusercontent.com/ju4n1t0x/subtes/main/Datasets/formaciones-despachadas-2025.csv"

formaciones_2025 = pd.read_csv(
    url_2025,
    sep=None,
    engine="python",
    encoding="latin1"
)

In [7]:
# Carga de los registros operativos históricos y de 2026.
# Se mantienen en DataFrames separados hasta verificar si sus estructuras son compatibles.

# Datos históricos (2015-2021)
url_historico = "https://raw.githubusercontent.com/ju4n1t0x/subtes/main/Datasets/formaciones-despachadas-total.csv"

formaciones_historico = pd.read_csv(
    url_historico,
    sep=None,
    engine="python",
    encoding="latin1"
)

# Datos de 2026
url_2026 = "https://raw.githubusercontent.com/ju4n1t0x/subtes/main/Datasets/formaciones-despachadas-2026.csv"

formaciones_2026 = pd.read_csv(
    url_2026,
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

In [8]:
# Carga de los conjuntos de datos complementarios.

base_url = "https://raw.githubusercontent.com/ju4n1t0x/subtes/main/Datasets/"


# Información sobre el estado de la flota.

estado_flota = pd.read_csv(
    base_url + "estado-de-flota.csv",
    sep=";"
)

# Información sobre la cantidad de pasajeros.
viajes_linea = pd.read_csv(
    base_url + "total_viajes_por_linea.csv",
    sep=";"
)
viajes_anual = pd.read_csv(base_url + "viajes_anual.csv")
lineas = pd.read_csv(base_url + "lineas-de-subte.csv")
estaciones = pd.read_csv(base_url + "BaseUnificadaEstaciones.csv")
estaciones_accesibles = pd.read_csv(base_url + "estaciones-accesibles.csv")

In [9]:
# Corrección del nombre de la columna y revisión inicial del dataset de estado de flota.

estado_flota.rename(columns={"aÃ±o": "año"}, inplace=True)
display(estado_flota.head())


,mes,año,linea,flota,en_servicio,reserva,rg,mantenimiento_estacionamiento,alistamiento
0,1,2024,A,CNR,15,,,4.0,
1,1,2024,B,Mitsub.,10,,2,4.0,
2,1,2024,NaN,CAF 6000,6,,3,4.0,
3,1,2024,C,CNR,7,,,4.0,
4,1,2024,NaN,Nagoya 5000,,,1,3.0,


# 3.3. Verificación y exploración inicial de los datos.

In [10]:
# Inspección de los primeros registros del dataset de 2025.


display(formaciones_2025.head())

print("Columnas del dataset:")
print(formaciones_2025.columns.tolist())

/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Fecha,Linea,Tipo Día,Registro,Orden,Tren,Formación A,Formación D,Modelo A,Modelo D,Causa A,Causa D,Cantidad coches A,Cantidad coches D,Km A,Km D,Tipo Viaje A,Tipo Viaje D,Hora sale A,Hora sale D
0,1/1/2025,A,Feriado,1.0,1.0,3.0,NaN,222,NaN,CNR,NaN,NaN,0,5,NaN,5.636,N,S,NaN,08:00:00
1,1/1/2025,A,Feriado,2.0,2.0,4.0,NaN,229,NaN,CNR,NaN,NaN,0,5,NaN,8.717,N,S,NaN,08:00:00
2,1/1/2025,A,Feriado,3.0,3.0,5.0,NaN,227,NaN,CNR,NaN,NaN,0,5,NaN,9.770,N,S,NaN,08:00:20
3,1/1/2025,A,Feriado,4.0,4.0,6.0,215,215,CNR,CNR,NaN,NaN,5,5,1.987,9.770,S,S,08:00:00,08:09:41
4,1/1/2025,A,Feriado,5.0,5.0,7.0,216,216,CNR,CNR,NaN,NaN,5,5,6.264,9.770,S,S,08:00:00,08:18:45


Columnas del dataset:
['Fecha', 'Linea', 'Tipo Día', 'Registro', 'Orden', 'Tren', 'Formación A', 'Formación D', 'Modelo A', 'Modelo D', 'Causa A', 'Causa D', 'Cantidad coches A', 'Cantidad  coches D', 'Km A', 'Km D', 'Tipo Viaje A', 'Tipo Viaje D', 'Hora sale A', 'Hora sale D']


In [11]:
# Comparación de dimensiones y columnas de los registros operativos de los distintos períodos.

datasets = {
    "Histórico (2015-2021)": formaciones_historico,
    "2025": formaciones_2025,
    "2026": formaciones_2026
}

for nombre, df in datasets.items():
    print(f"\n--- {nombre} ---")
    print("Dimensiones:", df.shape)
    print("Columnas:", df.columns.tolist())


--- Histórico (2015-2021) ---
Dimensiones: (751351, 18)
Columnas: ['fr1_fecha', 'fr1_linea', 'fr1_tipo', 'fr1_regist', 'fr1_orden', 'fr1_tren', 'fr1_cauc1', 'fr1_scauc1', 'fr1_cauc2', 'fr1_scauc2', 'fr1_cocc1', 'fr1_cocc2', 'fr1_km', 'fr1_kmv', 'fr1_viac1', 'fr1_viac2', 'fr1_salc1', 'fr1_salc2']

--- 2025 ---
Dimensiones: (495719, 20)
Columnas: ['Fecha', 'Linea', 'Tipo Día', 'Registro', 'Orden', 'Tren', 'Formación A', 'Formación D', 'Modelo A', 'Modelo D', 'Causa A', 'Causa D', 'Cantidad coches A', 'Cantidad  coches D', 'Km A', 'Km D', 'Tipo Viaje A', 'Tipo Viaje D', 'Hora sale A', 'Hora sale D']

--- 2026 ---
Dimensiones: (349166, 20)
Columnas: ['Fecha', 'Linea', 'Tipo Día', 'Registro', 'Orden', 'Tren', 'Formación A', 'Formación D', 'Modelo A', 'Modelo D', 'Causa A', 'Causa D', 'Cantidad coches A', 'Cantidad  coches D', 'Km A', 'Km D', 'Tipo Viaje A', 'Tipo Viaje D', 'Hora sale A', 'Hora sale D']


In [12]:
# Comparación de dimensiones y columnas de los conjuntos de datos complementarios.

datasets_complementarios = {
    "Estado de flota": estado_flota,
    "Viajes por línea": viajes_linea,
    "Viajes anuales": viajes_anual,
    "Líneas": lineas,
    "Estaciones": estaciones,
    "Estaciones accesibles": estaciones_accesibles
}

for nombre, df in datasets_complementarios.items():
    print(f"\n--- {nombre} ---")
    print("Dimensiones:", df.shape)
    print("Columnas:", df.columns.tolist())


--- Estado de flota ---
Dimensiones: (330, 9)
Columnas: ['mes', 'año', 'linea', 'flota', 'en_servicio', 'reserva', 'rg', 'mantenimiento_estacionamiento', 'alistamiento']

--- Viajes por línea ---
Dimensiones: (576, 7)
Columnas: ['periodo', 'mes', 'LINEA', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'pax_TOTAL']

--- Viajes anuales ---
Dimensiones: (48, 3)
Columnas: ['year', 'LINEA', 'total']

--- Líneas ---
Dimensiones: (82, 3)
Columnas: ['wkt', 'id', 'lineasub']

--- Estaciones ---
Dimensiones: (822363, 6)
Columnas: ['FECHA', 'ESTACION', 'LINEA', 'DESDE', 'HORA', 'CANTIDAD']

--- Estaciones accesibles ---
Dimensiones: (93, 6)
Columnas: ['long', 'lat', 'linea', 'estacion', 'escaleras_mecanicas', 'ascensores']


In [13]:
# Revisión de tipos de datos y cantidad de valores faltantes en todos los datasets.

todos_los_datasets = {
    **datasets,
    **datasets_complementarios
}

for nombre, df in todos_los_datasets.items():
    print(f"\n--- {nombre} ---")
    print("\nTipos de datos:")
    print(df.dtypes)
    print("\nValores faltantes:")
    print(df.isna().sum())


--- Histórico (2015-2021) ---

Tipos de datos:
fr1_fecha      object
fr1_linea      object
fr1_tipo       object
fr1_regist      int64
fr1_orden       int64
fr1_tren        int64
fr1_cauc1      object
fr1_scauc1     object
fr1_cauc2      object
fr1_scauc2     object
fr1_cocc1       int64
fr1_cocc2       int64
fr1_km        float64
fr1_kmv       float64
fr1_viac1      object
fr1_viac2      object
fr1_salc1      object
fr1_salc2      object
dtype: object

Valores faltantes:
fr1_fecha          0
fr1_linea          1
fr1_tipo           1
fr1_regist         0
fr1_orden          0
fr1_tren           0
fr1_cauc1     616261
fr1_scauc1    598252
fr1_cauc2     632401
fr1_scauc2    609273
fr1_cocc1          0
fr1_cocc2          0
fr1_km        520722
fr1_kmv       520722
fr1_viac1          1
fr1_viac2          1
fr1_salc1          1
fr1_salc2          1
dtype: int64

--- 2025 ---

Tipos de datos:
Fecha                  object
Linea                  object
Tipo Día               object
Registro  

In [14]:
# Cálculo del porcentaje de valores faltantes en los registros operativos.

for nombre, df in {
    "2025": formaciones_2025,
    "2026": formaciones_2026
}.items():

    print(f"\n--- {nombre} ---")

    faltantes = (
        df.isna()
        .mean()
        .mul(100)
        .sort_values(ascending=False)
    )

    display(faltantes[faltantes > 0].round(2))


--- 2025 ---


,0
Causa D,93.76
Causa A,93.59
Formación D,13.63
Modelo D,12.65
Formación A,9.79
Modelo A,9.79
Hora sale D,7.74
Km D,7.74
Cantidad coches D,7.59
Km A,7.02



--- 2026 ---


,0
Causa D,97.44
Causa A,96.71
Modelo D,9.87
Formación D,9.87
Km D,7.61
Hora sale D,7.61
Cantidad coches D,7.61
Formación A,5.89
Modelo A,5.89
Hora sale A,2.97


In [15]:
# Identificación de filas completamente vacías en los registros operativos.

for nombre, df in {
    "2025": formaciones_2025,
    "2026": formaciones_2026
}.items():

    filas_vacias = df.isna().all(axis=1).sum()

    print(f"{nombre}: {filas_vacias} filas completamente vacías")

2025: 24696 filas completamente vacías
2026: 0 filas completamente vacías


In [16]:
# Creación de versiones de trabajo sin filas completamente vacías.

formaciones_2025_limpio = formaciones_2025.dropna(how="all").copy()
formaciones_2026_limpio = formaciones_2026.dropna(how="all").copy()

print("2025:", formaciones_2025_limpio.shape)
print("2026:", formaciones_2026_limpio.shape)

2025: (471023, 20)
2026: (349166, 20)


In [17]:
# Exploración de los valores registrados en las columnas de causas.

for nombre, df in {
    "2025": formaciones_2025_limpio,
    "2026": formaciones_2026_limpio
}.items():

    print(f"\n--- {nombre} ---")

    print("\nCausa A:")
    display(df["Causa A"].value_counts(dropna=False).head(15))

    print("\nCausa D:")
    display(df["Causa D"].value_counts(dropna=False).head(15))


--- 2025 ---

Causa A:


,count
Causa A,
NaN,439241
,16236
Coche descompuesto,3724
Huelga / Paro General,1649
Falta de coches,1369
Obra de Modernización,1107
Falta de custodia policial,1015
Conflicto Gremial,999
Falta de custodia policial,647



Causa D:


,count
Causa D,
NaN,440100
,18881
Coche descompuesto,3404
Huelga / Paro General,1573
Obra de Modernización,1209
Conflicto Gremial,978
Falta de coches,875
Responsabilidad Siemens,589
Por Conductor,441



--- 2026 ---

Causa A:


,count
Causa A,
NaN,337676
Coche descompuesto,2672
Conflicto Gremial,1697
Huelga / Paro General,1390
Otros,763
Falta de coches,727
Obra de Modernización,645
Intento de suicidio /Arrollamiento,431
Vía bloqueada,422



Causa D:


,count
Causa D,
NaN,340238
Coche descompuesto,2520
Huelga / Paro General,1311
Conflicto Gremial,1272
Falta de coches,632
Obra de Modernización,536
Intento de suicidio /Arrollamiento,436
Por Conductor,294
Actitud / Desempeño de personal,220


In [18]:
# Diferenciación entre valores nulos y cadenas vacías en las columnas de causas.

for nombre, df in {
    "2025": formaciones_2025_limpio,
    "2026": formaciones_2026_limpio
}.items():

    print(f"\n--- {nombre} ---")

    for columna in ["Causa A", "Causa D"]:
        vacios = df[columna].eq("").sum()
        nulos = df[columna].isna().sum()

        print(f"{columna}:")
        print(f"  Valores nulos: {nulos}")
        print(f"  Cadenas vacías: {vacios}")


--- 2025 ---
Causa A:
  Valores nulos: 439241
  Cadenas vacías: 0
Causa D:
  Valores nulos: 440100
  Cadenas vacías: 0

--- 2026 ---
Causa A:
  Valores nulos: 337676
  Cadenas vacías: 0
Causa D:
  Valores nulos: 340238
  Cadenas vacías: 0


In [19]:
# Comparación de registros con y sin causa informada.

for nombre, df in {
    "2025": formaciones_2025_limpio,
    "2026": formaciones_2026_limpio
}.items():

    print(f"\n--- {nombre} ---")

    for columna in ["Causa A", "Causa D"]:
        con_causa = df[columna].notna()

        print(f"\n{columna}")
        print("Registros con causa:", con_causa.sum())
        print("Con formación:", df.loc[con_causa, "Formación A" if columna == "Causa A" else "Formación D"].notna().sum())
        print("Con modelo:", df.loc[con_causa, "Modelo A" if columna == "Causa A" else "Modelo D"].notna().sum())
        print("Con kilómetros:", df.loc[con_causa, "Km A" if columna == "Causa A" else "Km D"].notna().sum())


--- 2025 ---

Causa A
Registros con causa: 31782
Con formación: 18032
Con modelo: 18032
Con kilómetros: 31782

Causa D
Registros con causa: 30923
Con formación: 1691
Con modelo: 6580
Con kilómetros: 30923

--- 2026 ---

Causa A
Registros con causa: 11490
Con formación: 1282
Con modelo: 1282
Con kilómetros: 11490

Causa D
Registros con causa: 8928
Con formación: 1061
Con modelo: 1061
Con kilómetros: 8928


In [20]:
# Inspección de registros con causa informada y sin formación identificada.

columnas = [
    "Fecha", "Linea", "Tren",
    "Formación A", "Modelo A", "Causa A", "Km A",
    "Formación D", "Modelo D", "Causa D", "Km D"
]

for nombre, df in {
    "2025": formaciones_2025_limpio,
    "2026": formaciones_2026_limpio
}.items():

    print(f"\n--- {nombre}: Causa A sin formación ---")

    display(
        df.loc[
            df["Causa A"].notna() & df["Formación A"].isna(),
            columnas
        ].head(10)
    )

    print(f"\n--- {nombre}: Causa D sin formación ---")

    display(
        df.loc[
            df["Causa D"].notna() & df["Formación D"].isna(),
            columnas
        ].head(10)
    )


--- 2025: Causa A sin formación ---


,Fecha,Linea,Tren,Formación A,Modelo A,Causa A,Km A,Formación D,Modelo D,Causa D,Km D
3759,17/1/2025,A,18.0,NaN,NaN,Obra de Modernización,9.77,NaN,NaN,Obra de Modernización,9.77
3761,17/1/2025,A,1.0,NaN,NaN,Obra de Modernización,9.77,NaN,NaN,Obra de Modernización,9.77
4986,23/1/2025,A,7.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,216,CNR,NaN,9.77
4987,23/1/2025,A,8.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4988,23/1/2025,A,12.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4989,23/1/2025,A,15.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4991,23/1/2025,A,1.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,212,CNR,NaN,9.77
4996,23/1/2025,A,4.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,220,CNR,NaN,9.77
4998,23/1/2025,A,14.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
5001,23/1/2025,A,8.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77



--- 2025: Causa D sin formación ---


,Fecha,Linea,Tren,Formación A,Modelo A,Causa A,Km A,Formación D,Modelo D,Causa D,Km D
3759,17/1/2025,A,18.0,NaN,NaN,Obra de Modernización,9.77,NaN,NaN,Obra de Modernización,9.77
3761,17/1/2025,A,1.0,NaN,NaN,Obra de Modernización,9.77,NaN,NaN,Obra de Modernización,9.77
4983,23/1/2025,A,4.0,213,CNR,NaN,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4984,23/1/2025,A,5.0,216,CNR,NaN,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4987,23/1/2025,A,8.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4988,23/1/2025,A,12.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4989,23/1/2025,A,15.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4993,23/1/2025,A,13.0,226,CNR,NaN,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
4998,23/1/2025,A,14.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77
5001,23/1/2025,A,8.0,NaN,NaN,Pasajero descompuesto en formacion,9.77,NaN,NaN,Pasajero descompuesto en formacion,9.77



--- 2026: Causa A sin formación ---


,Fecha,Linea,Tren,Formación A,Modelo A,Causa A,Km A,Formación D,Modelo D,Causa D,Km D
227,2/1/2026,A,1,NaN,NaN,Obra de Modernización,9.77,NaN,NaN,Obra de Modernización,9.770
358,2/1/2026,A,7,NaN,NaN,Obra de Modernización,9.77,NaN,NaN,Obra de Modernización,9.770
2245,12/1/2026,A,2,NaN,NaN,Por Guarda,9.77,NaN,NaN,Por Guarda,9.770
4625,22/1/2026,A,2,NaN,NaN,Por Conductor,9.77,NaN,NaN,Por Conductor,9.770
5897,27/1/2026,A,10,NaN,NaN,Coche descompuesto,9.77,NaN,NaN,Coche descompuesto,9.770
6307,29/1/2026,A,15,NaN,NaN,Incidente con pasajeros,9.77,NaN,NaN,Incidente con pasajeros,9.770
6318,29/1/2026,A,12,NaN,NaN,Conflicto Gremial,9.77,NaN,NaN,Conflicto Gremial,9.770
6321,29/1/2026,A,1,NaN,NaN,Conflicto Gremial,9.77,NaN,NaN,Conflicto Gremial,9.770
6391,29/1/2026,A,2,NaN,NaN,Conflicto Gremial,9.77,NaN,NaN,Conflicto Gremial,9.770
6442,29/1/2026,A,7,NaN,NaN,Conflicto Gremial,9.77,NaN,NaN,Conflicto Gremial,9.770



--- 2026: Causa D sin formación ---


,Fecha,Linea,Tren,Formación A,Modelo A,Causa A,Km A,Formación D,Modelo D,Causa D,Km D
227,2/1/2026,A,1,NaN,NaN,Obra de Modernización,9.77,NaN,NaN,Obra de Modernización,9.770
358,2/1/2026,A,7,NaN,NaN,Obra de Modernización,9.77,NaN,NaN,Obra de Modernización,9.770
2245,12/1/2026,A,2,NaN,NaN,Por Guarda,9.77,NaN,NaN,Por Guarda,9.770
4625,22/1/2026,A,2,NaN,NaN,Por Conductor,9.77,NaN,NaN,Por Conductor,9.770
5141,23/1/2026,A,5,223,CNR,NaN,9.77,NaN,NaN,Coche descompuesto,9.770
5281,24/1/2026,A,5,218,CNR,NaN,9.77,NaN,NaN,Coche descompuesto,9.770
5897,27/1/2026,A,10,NaN,NaN,Coche descompuesto,9.77,NaN,NaN,Coche descompuesto,9.770
6307,29/1/2026,A,15,NaN,NaN,Incidente con pasajeros,9.77,NaN,NaN,Incidente con pasajeros,9.770
6318,29/1/2026,A,12,NaN,NaN,Conflicto Gremial,9.77,NaN,NaN,Conflicto Gremial,9.770
6321,29/1/2026,A,1,NaN,NaN,Conflicto Gremial,9.77,NaN,NaN,Conflicto Gremial,9.770


In [21]:
# Exploración de los valores de kilómetros en los registros de 2025 y 2026.

for nombre, df in {
    "2025": formaciones_2025_limpio,
    "2026": formaciones_2026_limpio
}.items():

    print(f"\n--- {nombre} ---")

    for columna in ["Km A", "Km D"]:
        print(f"\n{columna}")
        print("Valores más frecuentes:")
        display(df[columna].value_counts(dropna=False).head(10))

        print("Estadísticas:")
        display(df[columna].describe())


--- 2025 ---

Km A
Valores más frecuentes:


,count
Km A,
4.28,74114
9.77,73398
8.11,68971
10.46,64006
11.85,62613
11.83,48006
14.36,14175
13.20,13015
NaN,10088


Estadísticas:


,Km A
count,460935.000000
mean,728.652589
std,2618.824546
min,1.038000
25%,8.110000
50%,10.460000
75%,11.850000
max,14360.000000



Km D
Valores más frecuentes:


,count
Km D,
4.280,81505
9.770,80997
8.110,75612
10.460,71007
11.850,69339
11.830,53307
0.000,18881
NaN,13660
5.011,384


... (6 outputs left out)


Km D
Valores más frecuentes:


,count
Km D,
9.770,45627
4.280,44441
8.110,41316
10.460,40944
11.850,39460
11.830,29931
NaN,26584
9770,8170
4280,7749


Estadísticas:


,Km D
count,322582
unique,258
top,9.770
freq,45627


In [22]:
# Inspección de los valores de Km D en 2026 que requieren revisión.

km_d_2026 = formaciones_2026_limpio["Km D"].dropna()

print("Valores más altos:")
display(
    km_d_2026.value_counts()
    .head(30)
)

print("\nValores que no contienen punto decimal:")
display(
    km_d_2026[
        ~km_d_2026.astype(str).str.contains(".", regex=False)
    ].value_counts().head(30)
)

Valores más altos:


,count
Km D,
9.770,45627
4.280,44441
8.110,41316
10.460,40944
11.850,39460
11.830,29931
9770,8170
4280,7749
8110,7361



Valores que no contienen punto decimal:


,count
Km D,
9770,8170
4280,7749
8110,7361
10460,7360
11850,7243
"4,28",6334
"9,77",6159
"10,46",5550
"8,11",5455


In [23]:
# Clasificación de los formatos presentes en Km D de 2026.

km_d = formaciones_2026_limpio["Km D"].dropna().astype(str)

formatos = {
    "Con punto decimal": km_d.str.contains(".", regex=False),
    "Con coma decimal": km_d.str.contains(",", regex=False),
    "Sin separador": ~km_d.str.contains(r"[.,]", regex=True)
}

for nombre, condicion in formatos.items():
    print(f"{nombre}: {condicion.sum()} registros")

print("\nEjemplos de valores sin separador:")
display(km_d[formatos["Sin separador"]].value_counts().head(20))

Con punto decimal: 245364 registros
Con coma decimal: 33303 registros
Sin separador: 43915 registros

Ejemplos de valores sin separador:


,count
Km D,
9770,8170
4280,7749
8110,7361
10460,7360
11850,7243
11830,5384
5011,36
3864,34
1432,26


In [24]:
# Comparación de los kilómetros de salida y llegada en 2026.

comparacion_km = formaciones_2026_limpio[
    ["Km A", "Km D"]
].copy()

print("Registros donde Km A y Km D coinciden:")
print(
    (comparacion_km["Km A"].astype(str) ==
     comparacion_km["Km D"].astype(str)).sum()
)

print("\nEjemplos donde los valores son diferentes:")
display(
    comparacion_km[
        comparacion_km["Km A"].astype(str) !=
        comparacion_km["Km D"].astype(str)
    ].head(15)
)

Registros donde Km A y Km D coinciden:
407

Ejemplos donde los valores son diferentes:


,Km A,Km D
0,NaN,5.011
1,NaN,8.717
2,NaN,9.770
3,2.508,9.770
4,6.264,9.770
5,9.770,9.770
6,9.770,9.770
7,9.770,9.770
8,9.770,9.770
9,9.770,9.770


In [25]:
# Inspección de registros completos para interpretar la estructura de los datos A y D.

display(
    formaciones_2026_limpio[
        [
            "Fecha", "Linea", "Tipo Día", "Registro",
            "Orden", "Tren",
            "Formación A", "Modelo A", "Km A", "Tipo Viaje A",
            "Hora sale A",
            "Formación D", "Modelo D", "Km D", "Tipo Viaje D",
            "Hora sale D"
        ]
    ].head(15)
)

/usr/local/lib/python3.13/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,Fecha,Linea,Tipo Día,Registro,Orden,Tren,Formación A,Modelo A,Km A,Tipo Viaje A,Hora sale A,Formación D,Modelo D,Km D,Tipo Viaje D,Hora sale D
0,1/1/2026,A,Feriado,1,1,3,NaN,NaN,NaN,N,NaN,223,CNR,5.011,S,08:00:00
1,1/1/2026,A,Feriado,2,2,4,NaN,NaN,NaN,N,NaN,222,CNR,8.717,S,08:00:00
2,1/1/2026,A,Feriado,3,3,5,NaN,NaN,NaN,N,NaN,217,CNR,9.770,S,08:00:15
3,1/1/2026,A,Feriado,4,4,6,221,CNR,2.508,S,08:00:00,221,CNR,9.770,S,08:09:27
4,1/1/2026,A,Feriado,5,5,7,216,CNR,6.264,S,08:00:00,216,CNR,9.770,S,08:18:54
5,1/1/2026,A,Feriado,6,6,1,219,CNR,9.770,S,08:00:18,219,CNR,9.770,S,08:26:42
6,1/1/2026,A,Feriado,7,7,2,226,CNR,9.770,S,08:07:54,226,CNR,9.770,S,08:34:23
7,1/1/2026,A,Feriado,8,8,3,223,CNR,9.770,S,08:15:34,223,CNR,9.770,S,08:41:59
8,1/1/2026,A,Feriado,9,9,4,222,CNR,9.770,S,08:23:00,222,CNR,9.770,S,08:49:31
9,1/1/2026,A,Feriado,10,10,5,217,CNR,9.770,S,08:30:53,217,CNR,9.770,S,08:57:32


In [26]:
# Revisión de la actividad diaria de las formaciones identificadas.

formaciones_dia = (
    formaciones_2026_limpio
    .groupby(["Fecha", "Formación A"], dropna=True)
    .agg(
        cantidad_registros=("Registro", "count"),
        primer_horario=("Hora sale A", "min"),
        ultimo_horario=("Hora sale A", "max"),
        kilometros=("Km A", "count")
    )
    .reset_index()
)

display(formaciones_dia.head(15))

,Fecha,Formación A,cantidad_registros,primer_horario,ultimo_horario,kilometros
0,1/1/2026,102,1,09:01:06,09:01:06,1
1,1/1/2026,106,2,20:49:37,21:56:03,2
2,1/1/2026,109,11,08:09:42,21:01:34,11
3,1/1/2026,112,12,08:00:00,21:09:37,12
4,1/1/2026,113,10,08:34:19,19:19:38,10
5,1/1/2026,115,13,08:00:00,21:36:41,13
6,1/1/2026,116,12,08:22:04,21:28:47,12
7,1/1/2026,12,11,08:00:00,20:34:00,11
8,1/1/2026,13,12,08:13:00,21:00:00,12
9,1/1/2026,17,9,08:26:00,20:21:00,9


In [27]:
# Verificación de los valores exactos de fecha y formación.

print("Fechas disponibles:")
print(formaciones_2026_limpio["Fecha"].head(10).tolist())

print("\nRegistros de la formación 204:")
display(
    formaciones_2026_limpio[
        formaciones_2026_limpio["Formación A"].astype("string").str.strip() == "204"
    ][["Fecha", "Formación A", "Km A", "Hora sale A"]].head(10)
)

Fechas disponibles:
['1/1/2026', '1/1/2026', '1/1/2026', '1/1/2026', '1/1/2026', '1/1/2026', '1/1/2026', '1/1/2026', '1/1/2026', '1/1/2026']

Registros de la formación 204:


,Fecha,Formación A,Km A,Hora sale A
12820,1/1/2026,204,4.28,08:01:21
12824,1/1/2026,204,4.28,08:28:10
12828,1/1/2026,204,4.28,08:54:31
12832,1/1/2026,204,4.28,09:20:39
12836,1/1/2026,204,4.28,09:50:29
12840,1/1/2026,204,4.28,10:18:47
12844,1/1/2026,204,4.28,10:47:11
12848,1/1/2026,204,4.28,11:14:55
12852,1/1/2026,204,4.28,11:41:42
12856,1/1/2026,204,4.28,12:08:22


In [28]:
# Comparación de los kilómetros A y D para la formación 204 durante el día.

display(
    formaciones_2026_limpio[
        (formaciones_2026_limpio["Fecha"] == "1/1/2026") &
        (formaciones_2026_limpio["Formación A"].astype("string").str.strip() == "204")
    ][
        ["Linea", "Tren", "Formación A", "Km A",
         "Formación D", "Km D", "Hora sale A", "Hora sale D"]
    ].head(20)
)


,Linea,Tren,Formación A,Km A,Formación D,Km D,Hora sale A,Hora sale D
12820,C,3,204,4.28,204,4.280,08:01:21,08:15:14
12824,C,3,204,4.28,204,4.280,08:28:10,08:41:04
12828,C,3,204,4.28,204,4.280,08:54:31,09:07:22
12832,C,3,204,4.28,204,4.280,09:20:39,09:33:47
12836,C,3,204,4.28,204,4.280,09:50:29,10:04:28
12840,C,3,204,4.28,204,4.280,10:18:47,10:33:00
12844,C,3,204,4.28,204,4.280,10:47:11,11:00:19
12848,C,3,204,4.28,204,4.280,11:14:55,11:28:41
12852,C,3,204,4.28,204,4.280,11:41:42,11:54:48
12856,C,3,204,4.28,204,4.280,12:08:22,12:22:28


In [29]:
# Resumen de actividad de cada formación durante 2026.

actividad_formaciones = (
    formaciones_2026_limpio
    .dropna(subset=["Formación A"])
    .groupby("Formación A")
    .agg(
        dias_operativos=("Fecha", "nunique"),
        cantidad_registros=("Registro", "count"),
        modelo=("Modelo A", "first")
    )
    .sort_values("cantidad_registros", ascending=False)
    .reset_index()
)

display(actividad_formaciones.head(20))

,Formación A,dias_operativos,cantidad_registros,modelo
0,210,217,6534,CNR
1,211,205,6045,CNR
2,203,210,5902,CNR
3,202,203,5869,CNR
4,209,204,5828,CNR
5,207,197,5549,CNR
6,208,207,5463,CNR
7,Q,219,5256,Mitsubishi
8,306,214,4463,Alstom
9,K,233,4446,Mitsubishi


In [30]:
# Evaluación de la relación entre causas registradas e identificación de formaciones.

for columna_causa, columna_formacion in [
    ("Causa A", "Formación A"),
    ("Causa D", "Formación D")
]:
    causa_informada = (
        formaciones_2026_limpio[columna_causa].notna()
    )

    formacion_identificada = (
        formaciones_2026_limpio[columna_formacion].notna()
    )

    ambas = causa_informada & formacion_identificada

    print(f"\n{columna_causa} y {columna_formacion}")
    print("Registros con causa:", causa_informada.sum())
    print("Registros con formación:", formacion_identificada.sum())
    print("Registros con ambas:", ambas.sum())
    print(
        "Porcentaje de causas asociadas a una formación:",
        round(ambas.sum() / causa_informada.sum() * 100, 2),
        "%"
    )


Causa A y Formación A
Registros con causa: 11490
Registros con formación: 328592
Registros con ambas: 1282
Porcentaje de causas asociadas a una formación: 11.16 %

Causa D y Formación D
Registros con causa: 8928
Registros con formación: 314715
Registros con ambas: 1061
Porcentaje de causas asociadas a una formación: 11.88 %


#(NOTA INTERNA)
### 3.4. Diagnóstico de los datos

| Lo que ya sabemos | Lo que todavía no sabemos |
|---|---|
| Los registros operativos de 2015-2021, 2025 y 2026 tienen estructuras diferentes. | Si podemos homologar sus variables e integrar los períodos. |
| El archivo de 2025 contenía 24.696 filas completamente vacías, que excluimos de la versión de trabajo. | Si existen otros problemas de calidad que debamos resolver antes del análisis. |
| Hay muchos valores faltantes en las columnas de causas, formaciones y modelos. | Si los valores faltantes en las causas significan que no hubo incidentes o que no se registró la información. |
| Las causas incluyen fallas mecánicas, como «Coche descompuesto», pero también interrupciones por obras, conflictos gremiales y otros motivos. | Qué causas representan efectivamente fallas mecánicas de las formaciones. |
| Solo el 11,16 % de los registros con causa en A y el 11,88 % en D tienen una formación identificada en el mismo lado. | Por qué muchas causas no tienen una formación asociada y si podemos establecer esa relación. |
| Las columnas `Km A` y `Km D` presentan formatos inconsistentes y valores atípicos. | Qué representan exactamente esos kilómetros y si permiten calcular el desgaste acumulado. |
| Podemos calcular la cantidad de registros y los días operativos de cada formación en 2026. | Si estos indicadores son suficientes para aproximar la intensidad de uso de cada formación. |
| Los datos contienen información sobre actividad operativa y algunas incidencias. | Si contamos con información suficiente para construir una variable objetivo que identifique fallas mecánicas y entrenar un modelo predictivo. |

# 4. Análisis Exploratorio de Datos

En esta sección se analizan los conjuntos de datos para identificar patrones operativos, estudiar la relación entre las incidencias y las formaciones, y evaluar la disponibilidad de variables que permitan aproximar su desgaste acumulado.

El análisis se organiza en función de los objetivos del proyecto y busca determinar qué información puede utilizarse para construir las variables del modelo predictivo.

## 4.1 Análisis de las variables de kilometraje (Km A y Km D)

In [32]:
# Análisis de la variabilidad de Km A y Km D según la línea.

for nombre, df in {
    "2025": formaciones_2025_limpio,
    "2026": formaciones_2026_limpio
}.items():

    print(f"\n--- {nombre} ---")

    for columna in ["Km A", "Km D"]:
        print(f"\n{columna}")

        resumen = (
            df.groupby("Linea")[columna]
            .agg(
                registros="count",
                valores_distintos="nunique"
            )
        )

        display(resumen)


--- 2025 ---

Km A


,registros,valores_distintos
Linea,,
A,80892,36
B,69352,51
C,81213,18
D,71124,37
E,53274,41
H,75354,27
P,29726,4



Km D


,registros,valores_distintos
Linea,,
A,82390,25
B,70779,31
C,81731,11
D,72544,30
E,54687,31
H,76351,14
P,18881,1



--- 2026 ---

Km A


,registros,valores_distintos
Linea,,
A,60040,37
B,51934,63
C,58322,20
D,54027,43
E,39020,54
H,53951,24
P,21506,6



Km D


,registros,valores_distintos
Linea,,
A,61114,52
B,52954,61
C,58691,12
D,55097,54
E,40046,57
H,54680,24
P,0,0


In [33]:
# Identificación de los valores de kilometraje más frecuentes por línea en 2026.

for columna in ["Km A", "Km D"]:
    print(f"\n--- Valores más frecuentes de {columna} por línea ---")

    for linea in sorted(formaciones_2026_limpio["Linea"].dropna().unique()):
        datos = formaciones_2026_limpio.loc[
            formaciones_2026_limpio["Linea"] == linea,
            columna
        ]

        print(f"\nLínea {linea}")
        display(datos.value_counts().head(5).to_frame("frecuencia"))


--- Valores más frecuentes de Km A por línea ---

Línea A


,frecuencia
Km A,
9.770,44826
9770.000,8030
10.000,6041
4.134,148
6.841,147



Línea B


,frecuencia
Km A,
11.850,38596
11850.000,7093
12.000,5189
8.444,172
4.715,153



Línea C


,frecuencia
Km A,
4.280,43955
4280.000,7665
4.000,6259
1.251,121
2.662,121


... (18 outputs left out)


Línea H


,frecuencia
Km D,
8.110,41316
8110,7361
"8,11",5455
5.011,184
6.407,100



Línea P


,frecuencia
Km D,


In [34]:
# Comparación de Km A y Km D con formatos numéricos homogeneizados.
# Esta transformación es únicamente exploratoria; no modifica los datos originales.

datos_km = formaciones_2026_limpio[["Linea", "Km A", "Km D"]].copy()

# Homogeneización de Km A.
datos_km["Km A normalizado"] = datos_km["Km A"].where(
    datos_km["Km A"] < 1000,
    datos_km["Km A"] / 1000
)

# Homogeneización de Km D.
km_d = pd.to_numeric(
    datos_km["Km D"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)

datos_km["Km D normalizado"] = km_d.where(
    km_d < 1000,
    km_d / 1000
)

# Comparación de ambos valores.
comparacion_km = datos_km.dropna(
    subset=["Km A normalizado", "Km D normalizado"]
).copy()

comparacion_km["diferencia"] = (
    comparacion_km["Km A normalizado"]
    - comparacion_km["Km D normalizado"]
).abs()

print("Registros comparables:", len(comparacion_km))
print(
    "Registros con valores iguales:",
    (comparacion_km["diferencia"] < 0.001).sum()
)

print("\nDiferencia entre Km A y Km D:")
display(comparacion_km["diferencia"].describe())

Registros comparables: 312216
Registros con valores iguales: 275085

Diferencia entre Km A y Km D:


,diferencia
count,312216.0
mean,0.783509
std,20.180487
min,0.0
25%,0.0
50%,0.0
75%,0.0
max,905.54


In [35]:
# Identificación de registros con diferencias entre Km A y Km D.

diferencias_km = comparacion_km[
    comparacion_km["diferencia"] >= 0.001
].copy()

print("Registros con diferencias:", len(diferencias_km))

print("\nCantidad de diferencias por línea:")
display(
    diferencias_km.groupby("Linea")
    .agg(
        registros=("diferencia", "count"),
        diferencia_media=("diferencia", "mean"),
        diferencia_maxima=("diferencia", "max")
    )
    .round(3)
)

print("\n20 registros con las mayores diferencias:")
display(
    diferencias_km.sort_values(
        "diferencia", ascending=False
    )[
        [
            "Linea", "Km A", "Km D",
            "Km A normalizado", "Km D normalizado",
            "diferencia"
        ]
    ].head(20)
)

Registros con diferencias: 37131

Cantidad de diferencias por línea:


,registros,diferencia_media,diferencia_maxima
Linea,,,
A,7003,10.619,639.879
B,6143,2.5,728.15
C,6706,0.621,477.72
D,6540,19.002,905.54
E,4740,3.103,606.17
H,5999,1.959,831.89



20 registros con las mayores diferencias:


,Linea,Km A,Km D,Km A normalizado,Km D normalizado,diferencia
281609,D,916.00,10460,916.00,10.46,905.54
236291,D,916.00,10.460,916.00,10.46,905.54
328607,D,916.00,10.460,916.00,10.46,905.54
345735,H,840.00,8.110,840.00,8.11,831.89
250794,H,8.11,840,8.11,840.0,831.89
34139,H,840.00,8.110,840.00,8.11,831.89
342467,H,840.00,8.110,840.00,8.11,831.89
116784,H,8.11,833,8.11,833.0,824.89
204875,H,8.11,833,8.11,833.0,824.89
250804,H,833.00,8.110,833.00,8.11,824.89


In [36]:
# Inspección de registros con diferencias extremas entre Km A y Km D.

indices = diferencias_km.nlargest(20, "diferencia").index

columnas = [
    "Fecha", "Linea", "Tren",
    "Formación A", "Formación D",
    "Km A", "Km D",
    "Hora sale A", "Hora sale D"
]

revision = formaciones_2026_limpio.loc[indices, columnas].copy()

revision["Km A normalizado"] = diferencias_km.loc[indices, "Km A normalizado"]
revision["Km D normalizado"] = diferencias_km.loc[indices, "Km D normalizado"]
revision["diferencia"] = diferencias_km.loc[indices, "diferencia"]

display(revision.sort_values("diferencia", ascending=False))

,Fecha,Linea,Tren,Formación A,Formación D,Km A,Km D,Hora sale A,Hora sale D,Km A normalizado,Km D normalizado,diferencia
236291,22/6/2026,D,15,111,NaN,916.00,10.460,19:07:13,19:37:26,916.00,10.46,905.54
281609,16/7/2026,D,3,325,330,916.00,10460,17:26:11,18:10:15,916.00,10.46,905.54
328607,13/8/2026,D,11,111,325,916.00,10.460,17:24:10,17:48:11,916.00,10.46,905.54
34139,22/1/2026,H,5,302,NaN,840.00,8.110,23:08:27,23:28:53,840.00,8.11,831.89
250794,30/6/2026,H,7,NaN,312,8.11,840,14:29:55,14:50:00,8.11,840.0,831.89
342467,14/8/2026,H,1,307,NaN,840.00,8.110,15:24:55,15:45:13,840.00,8.11,831.89
345735,28/8/2026,H,14,311,NaN,840.00,8.110,14:48:06,15:07:38,840.00,8.11,831.89
115647,26/3/2026,H,3,302,302,8.11,833,13:30:04,13:50:45,8.11,833.0,824.89
116784,31/3/2026,H,9,308,308,8.11,833,13:18:11,13:39:34,8.11,833.0,824.89
204875,30/5/2026,H,4,310,310,8.11,833,14:15:14,14:37:07,8.11,833.0,824.89


In [37]:
# Análisis de valores de kilometraje superiores a 100.
# Umbral exploratorio para identificar los valores extremos observados.

for columna_km, columna_formacion in [
    ("Km A", "Formación A"),
    ("Km D", "Formación D")
]:
    print(f"\n--- {columna_km} ---")

    datos = formaciones_2026_limpio.copy()

    # Conversión auxiliar para poder comparar formatos numéricos.
    km_numerico = pd.to_numeric(
        datos[columna_km].astype("string").str.replace(
            ",", ".", regex=False
        ),
        errors="coerce"
    )

    anomalos = datos.loc[
        km_numerico > 100,
        ["Linea", "Fecha", "Tren", columna_formacion]
    ].copy()

    anomalos["km_original"] = datos.loc[km_numerico > 100, columna_km]
    anomalos["km_numerico"] = km_numerico[km_numerico > 100]

    resumen = (
        anomalos.groupby(["Linea", "km_numerico"])
        .agg(
            registros=("km_numerico", "count"),
            formaciones_distintas=(columna_formacion, "nunique")
        )
        .sort_values("registros", ascending=False)
    )

    display(resumen.head(20))


--- Km A ---


registros  formaciones_distintas
Linea km_numerico                                  
A     9770.0            8030                     19
C     4280.0            7665                     11
H     8110.0            7248                     13
D     10460.0           7213                     21
B     11850.0           7093                     26
E     11830.0           5248                     18
P     14360.0           1395                     11
      13200.0           1375                     11
D     665.0              171                     16
A     293.0               94                     18
      649.0               65                     18
D     3864.0              44                     10
B     8444.0              33                     18
      4715.0              32                     16
      1581.0              32                     16
E     2226.0              31                     16
D     7128.0              31                     14
H     1703.0              31                     11
E     4735.0              31                     16
H     5294.0              31                     12


--- Km D ---


,,registros,formaciones_distintas
Linea,km_numerico,,
A,9770.0,8170,19
C,4280.0,7749,11
H,8110.0,7361,13
D,10460.0,7360,21
B,11850.0,7243,26
E,11830.0,5384,18
D,3864.0,34,7
H,5011.0,31,11
A,7783.0,26,15


In [39]:
# Inspección del dataset de líneas para identificar
# las columnas disponibles y localizar la información de longitud.

print("Dimensiones:", lineas.shape)

print("\nColumnas:")
print(lineas.columns.tolist())

print("\nPrimeros registros:")
display(lineas.head())

Dimensiones: (82, 3)

Columnas:
['wkt', 'id', 'lineasub']

Primeros registros:


,wkt,id,lineasub
0,MULTILINESTRING ((-58.4521256031295 -34.566215...,1,LINEA D
1,MULTILINESTRING ((-58.4564891346516 -34.562309...,2,LINEA D
2,MULTILINESTRING ((-58.4446681474258 -34.570012...,3,LINEA D
3,MULTILINESTRING ((-58.4350135329444 -34.575178...,4,LINEA D
4,MULTILINESTRING ((-58.4257114410852 -34.578422...,5,LINEA D


In [40]:
# Inspección de las geometrías disponibles por línea.
# Verificamos cuántos registros corresponden a cada trazado
# y si existen geometrías duplicadas.

print("Cantidad de registros por línea:")
display(lineas["lineasub"].value_counts().sort_index())

print("\nTipos de geometría:")
display(lineas["wkt"].str.split(" ").str[0].value_counts())

print("\nGeometrías duplicadas:", lineas["wkt"].duplicated().sum())

print("\nValores nulos:")
display(lineas[["wkt", "id", "lineasub"]].isna().sum())

Cantidad de registros por línea:


,count
lineasub,
LINEA A,17
LINEA B,16
LINEA C,8
LINEA D,15
LINEA E,15
LINEA H,11



Tipos de geometría:


,count
wkt,
MULTILINESTRING,82



Geometrías duplicadas: 0

Valores nulos:


,0
wkt,0
id,0
lineasub,0


In [41]:
# Instalación e importación de herramientas para trabajar con geometrías geográficas.
!pip install geopandas shapely pyproj

import geopandas as gpd
from shapely import wkt

In [43]:
# Conversión de las geometrías WKT a objetos geográficos.
# Las coordenadas originales están expresadas en longitud y latitud (EPSG:4326).

lineas_geo = gpd.GeoDataFrame(
    lineas.copy(),
    geometry=lineas["wkt"].apply(wkt.loads),
    crs="EPSG:4326"
)

print("Sistema de coordenadas:", lineas_geo.crs)
print("Cantidad de geometrías:", len(lineas_geo))
print("Tipos de geometría:")
display(lineas_geo.geom_type.value_counts())

Sistema de coordenadas: EPSG:4326
Cantidad de geometrías: 82
Tipos de geometría:


,count
MultiLineString,82


In [44]:
# Inspección de las geometrías por línea.
# Comprobamos si los distintos registros de una misma línea se superponen.

for nombre_linea, grupo in lineas_geo.groupby("lineasub"):
    print(f"\n{nombre_linea}")
    print("Cantidad de geometrías:", len(grupo))
    print("Longitud de cada geometría (en grados):")
    display(grupo.geometry.length.describe())


LINEA A
Cantidad de geometrías: 17
Longitud de cada geometría (en grados):


/tmp/ipykernel_992/240988514.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  display(grupo.geometry.length.describe())


,0
count,17.000000
mean,0.006070
std,0.001919
min,0.002787
25%,0.004828
50%,0.006050
75%,0.007580
max,0.008786



LINEA B
Cantidad de geometrías: 16
Longitud de cada geometría (en grados):


/tmp/ipykernel_992/240988514.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  display(grupo.geometry.length.describe())


,0
count,16.000000
mean,0.007808
std,0.002081
min,0.005029
25%,0.006283
50%,0.007499
75%,0.009224
max,0.011778


... (8 outputs left out)

,0
count,15.000000
mean,0.008184
std,0.004222
min,0.004957
25%,0.006050
50%,0.006602
75%,0.009052
max,0.021943



LINEA H
Cantidad de geometrías: 11
Longitud de cada geometría (en grados):


/tmp/ipykernel_992/240988514.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'length' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  display(grupo.geometry.length.describe())


,0
count,11.000000
mean,0.006856
std,0.001543
min,0.004154
25%,0.006356
50%,0.006716
75%,0.008153
max,0.009074


In [45]:
# Conversión de coordenadas geográficas a un sistema proyectado en metros.
# EPSG:32721 corresponde a UTM zona 21 sur, adecuada para Buenos Aires.

lineas_metros = lineas_geo.to_crs(epsg=32721)

# Cálculo de la longitud de cada geometría en kilómetros.
lineas_metros["longitud_km"] = lineas_metros.geometry.length / 1000

# Visualización de las longitudes calculadas.
display(
    lineas_metros[["lineasub", "longitud_km"]]
    .sort_values(["lineasub", "longitud_km"], ascending=[True, False])
)

,lineasub,longitud_km
69,LINEA A,0.817009
24,LINEA A,0.816990
46,LINEA A,0.801130
68,LINEA A,0.767564
47,LINEA A,0.721007
...,...,...
64,LINEA H,0.712419
65,LINEA H,0.708054
77,LINEA H,0.688185
70,LINEA H,0.498962


In [47]:
# Análisis de superposición entre geometrías de una misma línea.

resultados_superposicion = []

for nombre_linea, grupo in lineas_metros.groupby("lineasub"):

    geometrías = list(grupo.geometry)
    indices = list(grupo.index)

    for i in range(len(geometrías)):
        for j in range(i + 1, len(geometrías)):

            interseccion = geometrías[i].intersection(geometrías[j])
            longitud_compartida = interseccion.length / 1000

            if longitud_compartida > 0.001:
                resultados_superposicion.append({
                    "linea": nombre_linea,
                    "geometria_1": indices[i],
                    "geometria_2": indices[j],
                    "longitud_superpuesta_km": longitud_compartida
                })

columnas = [
    "linea", "geometria_1", "geometria_2",
    "longitud_superpuesta_km"
]

superposiciones = pd.DataFrame(
    resultados_superposicion,
    columns=columnas
)

print("Cantidad de pares con superposición:", len(superposiciones))

if superposiciones.empty:
    print("No se detectaron superposiciones superiores a 1 metro.")
else:
    display(superposiciones.sort_values(
        "longitud_superpuesta_km", ascending=False
    ))

Cantidad de pares con superposición: 0
No se detectaron superposiciones superiores a 1 metro.


In [48]:
# Cálculo de la longitud total de las geometrías por línea.
# Se suman las longitudes individuales expresadas en kilómetros.

longitud_lineas = (
    lineas_metros
    .groupby("lineasub")["longitud_km"]
    .sum()
    .reset_index()
    .rename(columns={
        "lineasub": "Linea",
        "longitud_km": "Longitud total (km)"
    })
)

display(longitud_lineas)

,Linea,Longitud total (km)
0,LINEA A,9.641448
1,LINEA B,11.767361
2,LINEA C,4.436501
3,LINEA D,10.460454
4,LINEA E,11.940054
5,LINEA H,7.929140


In [49]:
# Diccionario con las longitudes calculadas por línea.
# Se utilizará posteriormente para estimar la distancia recorrida.

longitud_por_linea = dict(
    zip(longitud_lineas["Linea"], longitud_lineas["Longitud total (km)"])
)

print(longitud_por_linea)

{'LINEA A': 9.641447712482227, 'LINEA B': 11.76736126639649, 'LINEA C': 4.436501055719426, 'LINEA D': 10.460453735777232, 'LINEA E': 11.940054312428375, 'LINEA H': 7.929140123837132}


In [50]:
# Inspección de registros reales de kilometraje por línea.
# Buscamos observar los valores de Km A y Km D junto con la fecha
# y la formación, sin aplicar todavía conversiones ni modificaciones.

columnas_muestra = [
    "Linea", "Fecha", "Tren",
    "Formación A", "Km A",
    "Formación D", "Km D"
]

for linea, grupo in formaciones_2026_limpio.groupby("Linea"):

    print(f"\n========== {linea} ==========")

    muestra = grupo[columnas_muestra].dropna(
        subset=["Km A", "Km D"],
        how="all"
    )

    display(
        muestra.sample(
            n=min(5, len(muestra)),
            random_state=42
        )
    )


========== A ==========


,Linea,Fecha,Tren,Formación A,Km A,Formación D,Km D
211949,A,16/6/2026,9,220,9.77,220,9.770
4065,A,19/1/2026,1,225,9.77,225,9.770
210686,A,11/6/2026,9,216,9.77,216,9.770
210588,A,10/6/2026,15,219,9.77,219,9.770
302127,A,3/8/2026,18,230,9.77,230,9.770



========== B ==========


,Linea,Fecha,Tren,Formación A,Km A,Formación D,Km D
310344,B,2/8/2026,7,K,11.85,K,11.850
315767,B,24/8/2026,1,D,11.85,D,11.850
131411,B,19/4/2026,8,L,11.85,L,11.850
266689,B,19/7/2026,5,P,11850.00,P,11850
175835,B,20/5/2026,0,NaN,NaN,M,11.850



========== C ==========


,Linea,Fecha,Tren,Formación A,Km A,Formación D,Km D
96899,C,31/3/2026,2,NaN,4.28,NaN,4.280
183917,C,23/5/2026,5,210,4.28,210,4.280
94577,C,20/3/2026,9,202,4.28,202,4.280
321065,C,14/8/2026,9,210,4.28,210,4.280
225795,C,9/6/2026,10,203,4.28,203,4.280


... (4 outputs left out)


========== H ==========


,Linea,Fecha,Tren,Formación A,Km A,Formación D,Km D
69008,H,12/2/2026,6,310,8.00,310,"8,11"
295043,H,16/7/2026,7,312,8110.00,312,8110
70554,H,22/2/2026,4,306,8.00,306,"8,11"
343076,H,18/8/2026,7,308,8.11,308,8.110
31881,H,11/1/2026,5,321,8.11,321,8.110



========== P ==========


,Linea,Fecha,Tren,Formación A,Km A,Formación D,Km D
253053,P,24/6/2026,4,21,13.20,NaN,NaN
205323,P,4/5/2026,4,14,13.20,NaN,NaN
73815,P,19/2/2026,1,NaN,13.00,NaN,NaN
348232,P,21/8/2026,6,NaN,14.36,NaN,NaN
207305,P,26/5/2026,2,12,13.20,NaN,NaN



### Resumen del análisis de las variables de kilometraje (4.1)

- Se encontraron distintos formatos para registrar los valores de `Km A` y `Km D`, incluyendo diferencias en el uso de puntos, comas y escalas numéricas.
- Los valores extremos aparecen en distintas líneas y formaciones, por lo que no se puede asumir que todos sean errores de carga.
- Al revisar los valores más frecuentes, observamos que se repiten para una misma línea, independientemente de la formación y la fecha del despacho.
- Los valores habituales de `Km A` y `Km D` son similares a la extensión de cada línea. Esto sugiere que podrían representar una distancia asociada al recorrido y no el kilometraje acumulado de cada formación. De todas formas, no contamos con una definición oficial que confirme esta interpretación.
- A partir del archivo geográfico `lineas`, calculamos la longitud de cada línea en kilómetros. Los resultados fueron relativamente próximos a los valores de referencia de la propuesta.
- También verificamos que no hubiera superposiciones lineales superiores a un metro entre las geometrías de una misma línea. El archivo no incluye información geográfica del Premetro (P).

**Conclusión:** el análisis permitió obtener las longitudes de referencia de las líneas y detectar limitaciones en las variables de kilometraje. Antes de utilizarlas para estimar la distancia recorrida por cada formación y calcular su desgaste acumulado, necesitamos confirmar qué representan exactamente `Km A` y `Km D` y cómo deben interpretarse sus distintos formatos.

# 4.2. Análisis de incidencias y causas de no despacho

In [38]:
# Identificación de las columnas relacionadas con causas y descripciones.
# Primero verificamos qué información textual tenemos disponible.

columnas_incidencias = [
    columna for columna in formaciones_2026_limpio.columns
    if "causa" in columna.lower() or "descrip" in columna.lower()
]

print("Columnas disponibles:")
print(columnas_incidencias)

print("\nCantidad de registros no nulos por columna:")

display(
    formaciones_2026_limpio[columnas_incidencias]
    .notna()
    .sum()
    .to_frame(name="registros_con_dato")
)

Columnas disponibles:
['Causa A', 'Causa D']

Cantidad de registros no nulos por columna:


,registros_con_dato
Causa A,11490
Causa D,8928


# 5. Limpieza y preparación de datos

In [31]:
# Normalización de campos de texto en los registros operativos.
# Se eliminan espacios al principio y al final y se convierten cadenas vacías en valores nulos.

def normalizar_texto(df):
    df = df.copy()

    columnas_texto = df.select_dtypes(include="object").columns

    for columna in columnas_texto:
        df[columna] = df[columna].str.strip()
        df[columna] = df[columna].replace("", pd.NA)

    return df


# Aplicación de la limpieza a los tres períodos.
formaciones_historico_limpio = normalizar_texto(formaciones_historico)
formaciones_2025_limpio = normalizar_texto(formaciones_2025_limpio)
formaciones_2026_limpio = normalizar_texto(formaciones_2026_limpio)